# Afternoon class 30/08 — Extra practice 09 SOLUTIONS: apply and map   (L02)

Executed in the lab image against the real `../data/sales.csv`. Every quoted
number is what it actually printed.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Extra practice 09 — apply and map. Run this once.
import pandas as pd

sales = pd.read_csv("../data/sales.csv")

# A lookup that covers only two of the three categories.
margin_target = {"Technology": 0.25, "Furniture": 0.15}

print("categories in the data:", sorted(sales["Category"].unique()))
print("categories in the map: ", sorted(margin_target))

### Question 1

`Target` -> **`NaN` 209**, `0.25` 68, `0.15` 23. -> mapped `91`, unmapped `209`.

**70% of the file failed to map**, silently.

Without `dropna=False` the tally would read `0.25: 68, 0.15: 23` and total
91 — a tidy-looking summary of a 300-row file, with 209 rows simply absent
from it. Nothing raised, nothing warned, and the dtype stayed numeric.

This is the single most common way a category quietly disappears from a
report: a lookup written when the data had two categories, run later
against data that has three.

In [ ]:
sales["Target"] = sales["Category"].map(margin_target)
print(sales["Target"].value_counts(dropna=False))
print()
print("mapped:  ", sales["Target"].notna().sum())
print("unmapped:", sales["Target"].isna().sum())

### Question 2

present `['Furniture', 'Office Supplies', 'Technology']`, covered `['Furniture', 'Technology']` -> **uncovered `['Office Supplies']`**, `209` rows.

One line of set arithmetic, and it names the problem exactly.

And the missing category is the *largest* one — Office Supplies is 209 of
300 rows. That is not a coincidence you can rely on either way; the point is
that the size of the gap has nothing to do with how obvious it is. A lookup
missing 1 row and a lookup missing 209 look identical in the code.

Run this check every time you map a column through a hand-written
dictionary, and especially on anything scheduled — the failure mode is a
new value appearing months after the code was reviewed.

In [ ]:
present = set(sales["Category"])
covered = set(margin_target)
print("present:  ", sorted(present))
print("covered:  ", sorted(covered))
print("UNCOVERED:", sorted(present - covered))
print()
print("rows affected:", (sales["Category"] == "Office Supplies").sum())

### Question 3

`.fillna(0.10)` and `.get(c, 0.10)` -> identical results, `.equals()` is `True`.

Same output, different guarantees, exactly as in worksheet 09 Q6.
`fillna` repairs afterwards and cannot distinguish 'not in the lookup' from
'missing in the source'. `.get()` never creates the `NaN`.

Also note `0.10` here is an invented number standing in for a business
decision. Defaulting the target margin for the biggest category in the file
is not a formatting choice — it is a claim someone should sign off.

In [ ]:
a = sales["Category"].map(margin_target).fillna(0.10)
b = sales["Category"].map(lambda c: margin_target.get(c, 0.10))
print(a.value_counts())
print()
print("agree:", a.equals(b))

### Question 4

Margin describe -> mean `-0.0635`, min `-2.6311`, median `0.0712`, max `0.7158`. -> **`126` of `300`** orders beat their target.

The mean margin is **negative** while the median is positive. That gap is
the whole story: most orders make a small positive margin, and a minority
lose enough to drag the average below zero — the worst is `-2.63`, meaning
that order lost more than two and a half times its own revenue.

Quote the mean alone and the business looks unprofitable. Quote the median
alone and it looks fine. Neither is a lie and neither is sufficient; the
spread is the finding.

In [ ]:
sales["Target"] = sales["Category"].map(lambda c: margin_target.get(c, 0.10))
sales["Margin"] = sales["Profit"] / sales["Sales"]
print(sales["Margin"].describe().round(4).to_string())
print()
beat = sales["Margin"] > sales["Target"]
print("orders beating their target:", beat.sum(), "of", len(sales))

### Question 5

`apply` verdicts -> `missed 174`, `beat 126`. -> matches Q4's count exactly.

Two mechanisms, same answer, which is the check worth doing when you
rewrite a rule in a different style.

`axis=1` is what lets the function see whole rows, so `row["Margin"]` and
`row["Target"]` both resolve. Without it the function receives columns and
the expression means something else entirely.

In [ ]:
sales["Target"] = sales["Category"].map(lambda c: margin_target.get(c, 0.10))
sales["Margin"] = sales["Profit"] / sales["Sales"]

def verdict(row):
    return "beat" if row["Margin"] > row["Target"] else "missed"

sales["Verdict"] = sales.apply(verdict, axis=1)
print(sales["Verdict"].value_counts())
print()
print("matches Q4:", (sales["Verdict"] == "beat").sum() == (sales["Margin"] > sales["Target"]).sum())

### Question 6

`apply(axis=1)` `0.0019 s` vs vectorised `0.0001 s` -> about **20x**.

Twenty times slower on 300 rows, where both are instant and the choice does
not matter.

It matters at scale, and the ratio roughly holds: row-wise `apply` builds a
Series object per row and calls back into Python each time, while the
comparison runs once over the whole column in compiled code. On three
million rows that is the difference between well under a second and a
coffee break.

The rule from worksheet 09 Q3 stands: if operators can express it, use
operators. Keep `apply` for logic that genuinely cannot be vectorised.

In [ ]:
import time
sales["Target"] = sales["Category"].map(lambda c: margin_target.get(c, 0.10))
sales["Margin"] = sales["Profit"] / sales["Sales"]

def verdict(row):
    return "beat" if row["Margin"] > row["Target"] else "missed"

t0 = time.time(); sales.apply(verdict, axis=1); by_apply = time.time() - t0
t0 = time.time(); _ = sales["Margin"] > sales["Target"]; by_vector = time.time() - t0

print("apply(axis=1): %.4f s" % by_apply)
print("vectorised:    %.4f s" % by_vector)
print("ratio: %.0fx" % (by_apply / by_vector))

### Question 7

Region codes `ATL, NOR, NUN, ONT, PRA, QUE, WES, YUK`. -> the first five rows map to `ONT`.

`map` accepts a Series as well as a dict, matching on the Series' index.
That is useful when the lookup itself comes from data — another table, a
`groupby` result, a config file — rather than being typed by hand.

Note `PRA` is derived from `Prarie`, the misspelling. Codes generated from
dirty values inherit the dirt, and now the error is one step further from
its source.

In [ ]:
regions = sorted(sales["Region"].unique())
codes = pd.Series({r: r[:3].upper() for r in regions})
print(codes.to_string())
print()
print(sales["Region"].map(codes).head())

### Question 8

`Discount == 0` in **30** orders; `Profit == 0` in **0**. -> `1/Profit` runs fine; `1/Discount` **raises** `ZeroDivisionError: float division by zero`.

The same expression, the same function, two columns — one works and one
stops everything. The code is not what decided it; the data was.

That is the honest summary of this whole sheet. `apply` will run whatever
you give it against whatever is actually in the column, and 'it worked when
I tested it' means your test data happened to contain no zeros. Thirty rows
out of three hundred were enough here; one would have been.

Compare with Q1, where `map` met values it could not handle and returned
`NaN` for 209 of them without a murmur. Loud failure costs you a run and
points at the problem. Quiet failure costs you a report. Know which one you
have chosen.

In [ ]:
print("orders with Discount exactly 0:", (sales["Discount"] == 0).sum())
print("orders with Profit   exactly 0:", (sales["Profit"] == 0).sum())
print()
# Against Profit this runs perfectly -- there are no zeros in it.
print("1/Profit works, first 3:", sales["Profit"].apply(lambda p: 1 / p).head(3).round(4).tolist())
print()
print(sales["Discount"].apply(lambda d: 1 / d))